[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/compute_module1/blob/main/notebooks/lab_09_the_engine.ipynb)

# Lab 9: The Engine

*From* **A First Taste of Algorithms — and the Machines That Learn** *(The Curious Family Library). A computational lab: run the cells in order. A few labs end with a deliberately broken “Break it” cell — that one is meant to fail, and seeing how is the point.*

*The capstone. You build backpropagation from nothing — sixty lines — verify it against the calculus by hand and against finite differences by machine, then train a tiny model end to end with your own engine to prove it is real. Then numpy arrives as the scale instrument, the name model trains until it speaks, and the parrot exam from Chapter 8 returns with teeth: you watch the training loss fall while the held-out loss begins to climb, and you name the moment they part. Everything in nine chapters has been parts on a workbench. This is the build.*

## 1. The number that remembers

A plain number forgets its history. Build one that doesn't — the `Value`, learned in pieces. Start with the skeleton and the two core operations:

In [ ]:
import math, random

class Value:
    def __init__(self, data, parents=()):
        self.data = data
        self.grad = 0.0
        self._parents = set(parents)
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

a = Value(2.0)
b = Value(-3.0)
c = a * b
print(c.data, c._parents)

`Value(2.0)` builds a number-that-remembers; `c = a * b` builds a *new* one that remembers a and b as parents. The `isinstance` guard lets you write `a + 1` (it wraps the bare `1` into a Value). Each operation's `_backward` is a tiny function holding that operation's *derivative* — addition passes blame whole, multiplication scales it by the other factor. Confirm `c` knows its parents before moving on.

## 2. The backward sweep

History recorded forward; now price it backward. The sweep is Chapter 5's depth-first traversal, walked in reverse:

In [ ]:
    def backward(self):
        order = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for parent in v._parents:
                    build(parent)
                order.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

Add this method inside the `Value` class (same indentation as `__add__`). Then the moment of truth — build a small expression, sweep, and check against calculus you can do by hand:

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(7.0)
L = (a * b + c) * (a * b + c)        # L = (ab + c)^2
L.backward()
print("L    =", L.data, "  (expect 1)")
print("a.grad =", a.grad, "  (expect -6)")
print("b.grad =", b.grad, "  (expect  4)")
print("c.grad =", c.grad, "  (expect  2)")

By hand: ab + c = 1, so L = 1; the outer square contributes 2·(ab+c) = 2; then ∂L/∂a = 2·b = −6, ∂L/∂b = 2·a = 4, ∂L/∂c = 2. The engine should agree exactly. It does — and you have just watched blame flow from one number out to three, through a graph the computation recorded itself.

## 3. More operations, tested against reality

The model needs squashing and probabilities. Add these inside the class, each three lines, each just its derivative:

In [ ]:
    def __pow__(self, p):
        out = Value(self.data ** p, (self,))
        def _backward():
            self.grad += p * (self.data ** (p - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,))
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,))
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    def __neg__(self): return self * -1
    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

Now verify the *whole engine* the way professionals verify gradients: against **finite differences**. Calculus says the derivative is the limit of (f(x+h) − f(x−h)) / 2h; compute that numerically and check the engine matches:

In [ ]:
def f(av, bv, cv):
    return (math.tanh(av * bv) + cv ** 2) * (av + cv)

a = Value(0.5); b = Value(1.3); c = Value(-0.7)
out = ((a * b).tanh() + c ** 2) * (a + c)
out.backward()

h = 1e-6
ga = (f(0.5 + h, 1.3, -0.7) - f(0.5 - h, 1.3, -0.7)) / (2 * h)
print("a.grad engine:", round(a.grad, 5), "  finite diff:", round(ga, 5))

They agree to five decimals — your sixty lines compute the same derivatives the limit does, on an expression you'd never want to differentiate by hand. This is the test that makes the engine *trustworthy*: not a proof, but the check every practitioner actually runs.

## 4. A tiny model, trained by your own engine

Before numpy, prove the engine can *learn* — train a tiny next-letter model with nothing but Values and your backward sweep. Small enough to be pure Value, real enough to work:

In [ ]:
random.seed(0)
names = ["ada", "gauss", "euler", "abel", "nash", "hardy", "klein", "peano", "boole", "bayes"]
chars = sorted(set("".join(names))) + ["."]
stoi = {c: i for i, c in enumerate(chars)}
V = len(chars)
